In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Three-Way Entity Resolution: TKPI + MyFCD + USDA
Konversi dari notebook entity_resolution_threeway.ipynb ke skrip Python.
 
Cara pakai:
  1. Sesuaikan BASE, TKPI_PATH, MYFCD_PATH, dan OUTPUT_DIR di bagian PATHS.
  2. Jalankan:  python entity_resolution_threeway.py
"""
"""
Three-Way Entity Resolution: TKPI + MyFCD + USDA
=================================================
Links Southeast Asian food items (TKPI, MyFCD) to their closest
USDA SR Legacy equivalents so that USDA nutrient values can be
borrowed to fill schema gaps (magnesium, total sugars, saturated fat,
cholesterol, vitamin B6, vitamin B12).
 
Pipeline
--------
Step 1 : Category crosswalk  — map each regional category to USDA categories
Step 2 : Text normalisation  — clean food names, keep English translations
Step 3 : TF-IDF cosine match — score similarity within category blocks
Step 4 : Confidence labelling — HIGH / MEDIUM / LOW per match
Step 5 : Borrowing policy    — decide what to borrow based on confidence
Step 6 : Value borrowing     — pull USDA nutrient values into the linked records
 
Usage
-----
Set BASE, TKPI_PATH, MYFCD_PATH to your local paths, then run:
    python entity_resolution.py
"""
 
import pandas as pd
import numpy as np
import re
from collections import Counter
 
# ─────────────────────────────────────────────────────────────────────────────
# 0. PATHS  ── adjust to your folder layout
# ─────────────────────────────────────────────────────────────────────────────
BASE       = r"C:\Users\intan\OneDrive\Documents\Intan\02\2026\Jatim Melaju\Dataset\USDA\FoodDataLegacy2018"
TKPI_PATH  = r"C:\Users\intan\OneDrive\Documents\Intan\02\2026\Jatim Melaju\Dataset\GitHub\food-nutrition\data_clean\tkpi_clean.csv"
MYFCD_PATH = r"C:\Users\intan\OneDrive\Documents\Intan\02\2026\Jatim Melaju\Dataset\GitHub\food-nutrition\data_clean\myfcd_clean.csv"
 
# Folder tempat menyimpan semua output CSV (ubah bila perlu; default = folder USDA/BASE)
import os
OUTPUT_DIR = BASE
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────────────────────────────────────
tkpi  = pd.read_csv(TKPI_PATH)
myfcd = pd.read_csv(MYFCD_PATH)
 
usda  = pd.read_csv(f"{BASE}/food.csv")
ucat  = pd.read_csv(f"{BASE}/food_category.csv")
usda  = usda.merge(
    ucat[["id", "description"]].rename(
        columns={"id": "food_category_id", "description": "usda_category"}
    ),
    on="food_category_id", how="left"
)
 
food_nutrient = pd.read_csv(f"{BASE}/food_nutrient.csv")
nutrient_meta = pd.read_csv(f"{BASE}/nutrient.csv")

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. CATEGORY CROSSWALK
#    Each regional category maps to one or more USDA categories.
#    Purpose: restrict TF-IDF matching to foods of the same food group
#             (called "blocking") so we don't compare vegetables with meats.
# ─────────────────────────────────────────────────────────────────────────────
CROSSWALK_TKPI = {
    "serealia":              ["Cereal Grains and Pasta", "Baked Products",
                              "Breakfast Cereals"],
    "sayuran":               ["Vegetables and Vegetable Products"],
    "buah":                  ["Fruits and Fruit Juices"],
    "daging":                ["Beef Products", "Pork Products",
                              "Lamb, Veal, and Game Products",
                              "Poultry Products", "Sausages and Luncheon Meats"],
    "ikan/kerang/udang dll": ["Finfish and Shellfish Products"],
    "kacang-kacangan":       ["Legumes and Legume Products", "Nut and Seed Products"],
    "umbi berpati":          ["Vegetables and Vegetable Products",
                              "Cereal Grains and Pasta"],
    "telur":                 ["Dairy and Egg Products"],
    "susu":                  ["Dairy and Egg Products"],
    "minyak/lemak":          ["Fats and Oils"],
    "bumbu":                 ["Spices and Herbs", "Soups, Sauces, and Gravies"],
    "minuman non alkohol":   ["Beverages"],
    "konfeksioneri":         ["Sweets", "Snacks"],
}
 
CROSSWALK_MYFCD = {
    "cereals and grain products":          ["Cereal Grains and Pasta", "Baked Products"],
    "rice and rice flour based":           ["Cereal Grains and Pasta"],
    "wheat flour based":                   ["Baked Products", "Cereal Grains and Pasta"],
    "cereal based":                        ["Breakfast Cereals", "Cereal Grains and Pasta"],
    "glutinous rice based":                ["Cereal Grains and Pasta"],
    "vegetables and vegetable products":   ["Vegetables and Vegetable Products"],
    "vegetable dishes":                    ["Vegetables and Vegetable Products",
                                            "Soups, Sauces, and Gravies"],
    "vegetables and fruits based":         ["Vegetables and Vegetable Products",
                                            "Fruits and Fruit Juices"],
    "fruits and fruit products":           ["Fruits and Fruit Juices"],
    "meat and meat products":              ["Beef Products", "Pork Products",
                                            "Poultry Products",
                                            "Lamb, Veal, and Game Products"],
    "meat dishes":                         ["Beef Products", "Pork Products",
                                            "Poultry Products"],
    "fish, shellfish and products":        ["Finfish and Shellfish Products"],
    "fish and sea-food dishes":            ["Finfish and Shellfish Products"],
    "legumes and legume products":         ["Legumes and Legume Products"],
    "legume based":                        ["Legumes and Legume Products"],
    "nuts, seeds and products":            ["Nut and Seed Products"],
    "eggs":                                ["Dairy and Egg Products"],
    "milk and milk products":              ["Dairy and Egg Products"],
    "oils and fats":                       ["Fats and Oils"],
    "sugars and syrups":                   ["Sweets"],
    "beverages":                           ["Beverages"],
    "starchy roots, tubers and products":  ["Vegetables and Vegetable Products"],
    "tuber based":                         ["Vegetables and Vegetable Products"],
    "porridge and ipengat/i":              ["Cereal Grains and Pasta",
                                            "Breakfast Cereals"],
    "miscellaneous":                       ["Soups, Sauces, and Gravies",
                                            "Spices and Herbs",
                                            "Meals, Entrees, and Side Dishes"],
}

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. TEXT NORMALISATION
#    KEY DESIGN DECISION: keep parentheses content.
#    TKPI names follow the pattern  "Bahasa name (English name)"
#    e.g. "kacang arab, kering (chick pea, raw)"
#    Removing the brackets discards the English translation, which is the
#    only part that overlaps with USDA names. We keep it.
# ─────────────────────────────────────────────────────────────────────────────
STOPWORDS = {
    # English generic food-state words
    "raw", "cooked", "dried", "dry", "fresh", "canned", "frozen", "boiled",
    "grilled", "fried", "baked", "steamed", "smoked", "salted", "roasted",
    "per", "100g", "and", "or", "with", "the", "in", "of", "a", "an",
    "mature", "seeds", "products", "food", "foods", "grade", "whole",
    # Bahasa generic food-state words
    "mentah", "kering", "goreng", "rebus", "segar", "dan",
    "masak", "matang", "olahan", "panggang",
}
 
def normalize_name(text: str) -> str:
    """
    Lowercase → open brackets → remove punctuation → strip stopwords.
    Keeps English translations embedded in TKPI/MyFCD names.
    """
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[()\/,]", " ", text)   # open brackets and slashes
    text = re.sub(r"[^a-z\s]", " ", text)  # remove non-alpha
    tokens = [w for w in text.split()
              if w not in STOPWORDS and len(w) > 2]
    return " ".join(tokens)
 
tkpi ["name_norm"] = tkpi ["food_name_normalized"].apply(normalize_name)
myfcd["name_norm"] = myfcd["food_name_normalized"].apply(normalize_name)
usda ["name_norm"] = usda ["description"].apply(normalize_name)

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. TF-IDF COSINE SIMILARITY  (pure numpy — no sklearn required)
#    Why TF-IDF?  It up-weights rare, specific food terms (e.g. "arrowroot",
#    "rambutan") and down-weights common ones (e.g. "bean", "oil"),
#    giving more meaningful similarity scores than simple word overlap.
# ─────────────────────────────────────────────────────────────────────────────
def tfidf_cosine_match(
    query_names:  list,   # food names to match (TKPI or MyFCD block)
    corpus_names: list,   # candidate pool (USDA block)
    corpus_ids:   list,   # USDA fdc_id for each corpus entry
    top_n: int = 3        # return top-n candidates per query
) -> list:
    """
    Returns list of (query_index, usda_fdc_id, cosine_score) tuples.
    """
    all_docs  = query_names + corpus_names
    n_query   = len(query_names)
    tokenized = [d.split() for d in all_docs]
 
    # Build vocabulary from all documents in this block
    vocab     = sorted({w for doc in tokenized for w in doc})
    word_idx  = {w: i for i, w in enumerate(vocab)}
    V         = len(vocab)
 
    # Term Frequency matrix
    tf = np.zeros((len(all_docs), V), dtype=np.float32)
    for di, doc in enumerate(tokenized):
        cnt = Counter(doc)
        for w, c in cnt.items():
            if w in word_idx:
                tf[di, word_idx[w]] = c / max(len(doc), 1)
 
    # Inverse Document Frequency  (smoothed)
    df   = (tf > 0).sum(axis=0)
    idf  = np.log((len(all_docs) + 1) / (df + 1)) + 1
 
    # TF-IDF matrix, L2-normalised (so dot product = cosine similarity)
    tfidf      = tf * idf
    norms      = np.linalg.norm(tfidf, axis=1, keepdims=True)
    norms[norms == 0] = 1
    tfidf_norm = tfidf / norms
 
    # Cosine similarity: queries × corpus
    Q   = tfidf_norm[:n_query]
    C   = tfidf_norm[n_query:]
    sim = Q @ C.T                 # shape: (n_query, n_corpus)
 
    results = []
    for qi in range(n_query):
        top_idx = np.argsort(sim[qi])[::-1][:top_n]
        for ci in top_idx:
            results.append((qi, corpus_ids[ci], float(sim[qi, ci])))
    return results

In [23]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. CATEGORY-BLOCKED MATCHING
#    We only compare foods within the same mapped category block.
#    This is called "blocking" in record-linkage literature — it reduces
#    complexity from O(n×m) to O(n_block × m_block) and improves precision.
# ─────────────────────────────────────────────────────────────────────────────
def match_to_usda(
    source_df:      pd.DataFrame,
    source_id_col:  str,
    source_cat_col: str,
    crosswalk:      dict,
    usda_df:        pd.DataFrame,
    threshold:      float = 0.20,
    top_n:          int   = 3
) -> pd.DataFrame:
    """
    For every food in source_df, find its top-n USDA matches within the
    mapped category block and return all pairs above the threshold.
 
    threshold=0.20 is permissive — we keep LOW-confidence matches so
    they can be reviewed or used for category-level borrowing.
    Raise to 0.65 if you only want high-confidence direct borrowing.
    """
    all_links = []
    for src_cat, usda_cats in crosswalk.items():
        src_block  = source_df[source_df[source_cat_col] == src_cat].copy()
        usda_block = usda_df[usda_df["usda_category"].isin(usda_cats)].copy()
        if src_block.empty or usda_block.empty:
            continue
 
        q_names  = src_block["name_norm"].tolist()
        c_names  = usda_block["name_norm"].tolist()
        c_ids    = usda_block["fdc_id"].tolist()
        src_ids  = src_block[source_id_col].tolist()
        src_orig = src_block["food_name_normalized"].tolist()
 
        matches = tfidf_cosine_match(q_names, c_names, c_ids, top_n=top_n)
        print(matches)
 
        for qi, usda_fdc_id, score in matches:
            if score < threshold:
                continue
            usda_row = usda_df[usda_df["fdc_id"] == usda_fdc_id].iloc[0]
            all_links.append({
                "source_food_id":   src_ids[qi],
                "source_food_name": src_orig[qi],
                "source_category":  src_cat,
                "usda_fdc_id":      usda_fdc_id,
                "usda_food_name":   usda_row["description"],
                "usda_category":    usda_row["usda_category"],
                "similarity_score": round(score, 4),
            })
    return pd.DataFrame(all_links)
 
print("Matching TKPI → USDA ...")
tkpi_links  = match_to_usda(tkpi,  "food_id", "category_normalized",
                             CROSSWALK_TKPI,  usda, threshold=0.20)
 
print("Matching MyFCD → USDA ...")
myfcd_links = match_to_usda(myfcd, "food_id", "category_normalized",
                             CROSSWALK_MYFCD, usda, threshold=0.20)

Matching TKPI → USDA ...
[(0, 169742, 0.23722194568422636), (0, 168914, 0.23722194568422636), (0, 173900, 0.2332726236390852), (1, 169742, 0.16527453901707134), (1, 168914, 0.16527453901707134), (1, 173900, 0.16252301289431742), (2, 169742, 0.16254601604875105), (2, 168914, 0.16254601604875105), (2, 173900, 0.1598399149640491), (3, 169742, 0.24149824324249203), (3, 168914, 0.24149824324249203), (3, 173900, 0.2374777284745624), (4, 170288, 0.46812433035054746), (4, 170290, 0.41730210597507705), (4, 169695, 0.33897451560173664), (5, 168920, 0.2931423012655934), (5, 169748, 0.2547689799613331), (5, 169694, 0.19431396850432134), (6, 169711, 0.35587445827047953), (6, 168883, 0.297643060127306), (6, 169742, 0.12752198574818474), (7, 169711, 0.471132115911843), (7, 168883, 0.39404121719146673), (7, 169714, 0.22064702578242573), (8, 169742, 0.22447529858970833), (8, 168914, 0.22447529858970833), (8, 173900, 0.2207381854707979), (9, 169742, 0.1825897245714126), (9, 168914, 0.1825897245714126), 

[(0, 169250, 0.6346253567452287), (0, 168536, 0.596793477917662), (0, 168430, 0.5698135597227014), (1, 170551, 0.0), (1, 169237, 0.0), (1, 169247, 0.0), (2, 170551, 0.0), (2, 169237, 0.0), (2, 169247, 0.0), (3, 168412, 0.6864981002085417), (3, 170551, 0.0), (3, 169236, 0.0), (4, 170007, 0.4780232404070812), (4, 170414, 0.23825349522753123), (4, 169397, 0.2315046347081322), (5, 168414, 0.21639128715264905), (5, 168530, 0.20792184262770605), (5, 168415, 0.201541463508756), (6, 170414, 0.23825349522753123), (6, 169397, 0.2315046347081322), (6, 170415, 0.20355519970676617), (7, 168462, 0.6364365181085462), (7, 170531, 0.5420556683222161), (7, 168463, 0.48872952350543875), (8, 168462, 0.4610745963152), (8, 170531, 0.3926991794795247), (8, 168463, 0.3540663700871383), (9, 170375, 0.5289089006244768), (9, 168508, 0.493353511048047), (9, 170376, 0.4685053294214235), (10, 170402, 0.27609496482938856), (10, 169344, 0.2605956997812643), (10, 170403, 0.24943942792447815), (11, 170402, 0.3683915899

[(0, 174687, 0.0), (0, 168173, 0.0), (0, 168197, 0.0), (1, 174687, 0.0), (1, 168173, 0.0), (1, 168197, 0.0), (2, 168215, 0.2210992057893364), (2, 168216, 0.2210992057893364), (2, 168199, 0.2210992057893364), (3, 174687, 0.0), (3, 168173, 0.0), (3, 168197, 0.0), (4, 174687, 0.0), (4, 168173, 0.0), (4, 168197, 0.0), (5, 171715, 0.46778312065910127), (5, 174687, 0.0), (5, 168205, 0.0), (6, 174687, 0.0), (6, 168173, 0.0), (6, 168197, 0.0), (7, 174687, 0.0), (7, 168173, 0.0), (7, 168197, 0.0), (8, 174687, 0.0), (8, 168173, 0.0), (8, 168197, 0.0), (9, 174687, 0.0), (9, 168173, 0.0), (9, 168197, 0.0), (10, 169942, 0.6103313308015046), (10, 169941, 0.6103313308015046), (10, 169943, 0.265596514635734), (11, 174687, 0.0), (11, 168173, 0.0), (11, 168197, 0.0), (12, 174687, 0.0), (12, 168173, 0.0), (12, 168197, 0.0), (13, 169108, 0.6301854497595362), (13, 169109, 0.41569342247878754), (13, 169110, 0.40775032327898453), (14, 174687, 0.0), (14, 168173, 0.0), (14, 168197, 0.0), (15, 174687, 0.0), (15

[(0, 174193, 0.3337451870068601), (0, 174194, 0.31204609675998063), (0, 173681, 0.0), (1, 174193, 0.3774985099595599), (1, 174194, 0.35295471261183475), (1, 173681, 0.0), (2, 174223, 0.2925246446615436), (2, 171982, 0.2925246446615436), (2, 175181, 0.0), (3, 175181, 0.0), (3, 173681, 0.0), (3, 173695, 0.0), (4, 175181, 0.0), (4, 173681, 0.0), (4, 173695, 0.0), (5, 173698, 0.4067354090750962), (5, 173699, 0.38420089128405865), (5, 175181, 0.0), (6, 173675, 0.6608162461713578), (6, 171995, 0.608056766543318), (6, 175181, 0.0), (7, 175181, 0.0), (7, 173681, 0.0), (7, 173695, 0.0), (8, 175181, 0.0), (8, 173681, 0.0), (8, 173695, 0.0), (9, 174193, 0.30373091920236467), (9, 174194, 0.2839832647548286), (9, 173681, 0.0), (10, 174186, 0.2424875847014883), (10, 175165, 0.24040603581851358), (10, 174187, 0.233273353044095), (11, 175181, 0.0), (11, 173681, 0.0), (11, 173695, 0.0), (12, 175181, 0.0), (12, 173681, 0.0), (12, 173695, 0.0), (13, 175181, 0.0), (13, 173681, 0.0), (13, 173695, 0.0), (14

[(0, 173423, 0.24683044866117768), (0, 171287, 0.24683044866117768), (0, 172188, 0.24683044866117768), (1, 173423, 0.2632553089549685), (1, 171287, 0.2632553089549685), (1, 172188, 0.2632553089549685), (2, 173428, 0.38166665811798894), (2, 172184, 0.38166665811798894), (2, 173436, 0.32133327206990947), (3, 172183, 0.3573309474309563), (3, 172204, 0.3573309474309563), (3, 172203, 0.2964553845628434), (4, 172189, 0.38510736606541424), (4, 173423, 0.2055143201769692), (4, 171287, 0.2055143201769692), (5, 172189, 0.46246239207270423), (5, 173428, 0.45328039297228406), (5, 172184, 0.45328039297228406), (6, 172189, 0.4669416302865514), (6, 172183, 0.43588691457838), (6, 172204, 0.43588691457838), (7, 172189, 0.4910339609077993), (7, 173423, 0.2620425355422299), (7, 171287, 0.2620425355422299), (8, 173423, 0.19563321776209192), (8, 171287, 0.19563321776209192), (8, 172188, 0.19563321776209192), (9, 173423, 0.3062756866596848), (9, 171287, 0.3062756866596848), (9, 172188, 0.3062756866596848), 

[(0, 171867, 0.43050039327062756), (0, 171847, 0.4167534003354952), (0, 175016, 0.40908785177199897), (1, 172790, 0.3557291186960162), (1, 172738, 0.3557291186960162), (1, 171867, 0.34667201076128945), (2, 171867, 0.9436782778330064), (2, 174941, 0.2924812438685637), (2, 171866, 0.2764820410017426), (3, 174980, 0.33990205435903725), (3, 171867, 0.26586034110795487), (3, 174941, 0.20188404043735014), (4, 175084, 0.0), (4, 170289, 0.0), (4, 170688, 0.0), (5, 172680, 0.47126728726295075), (5, 172817, 0.4602313836688168), (5, 174075, 0.4007365128530438), (6, 167944, 0.3387422598301956), (6, 167925, 0.3121080435455591), (6, 174084, 0.28335634693017037), (7, 171847, 0.26335888142652053), (7, 175016, 0.2585147930673665), (7, 172782, 0.23326249572908492), (8, 175084, 0.0), (8, 170289, 0.0), (8, 170688, 0.0), (9, 167943, 0.4317962738728042), (9, 174099, 0.1899909415372862), (9, 171861, 0.16858638958215394), (10, 168891, 0.21481061031797485), (10, 168890, 0.2054313169918756), (10, 168889, 0.2010

[(0, 170881, 0.21701899829639898), (0, 172217, 0.20788759170545598), (0, 171302, 0.2071418655405877), (1, 171287, 0.2084774148016108), (1, 173423, 0.2084774148016108), (1, 172188, 0.2084774148016108)]
[(0, 173410, 0.42419070353819827), (0, 171314, 0.31672478605544235), (0, 173430, 0.2393032289368333), (1, 169081, 0.39890986580224047), (1, 171254, 0.37005249621470376), (1, 170896, 0.32122145050829015), (2, 172175, 0.6300782554207871), (2, 173418, 0.12383152288542994), (2, 169081, 0.09740997718429732), (3, 172177, 0.9240816810201111), (3, 173418, 0.0964127773013847), (3, 169081, 0.0758414838028907), (4, 172178, 0.9240816810201111), (4, 173418, 0.0964127773013847), (4, 169081, 0.0758414838028907), (5, 171295, 0.48373527800032246), (5, 173439, 0.43157184415708205), (5, 170896, 0.418836154931305), (6, 173418, 0.07071862706133589), (6, 169081, 0.055629614237426836), (6, 171251, 0.04814504465373066), (7, 173416, 0.9107945650873734), (7, 173439, 0.6203376188889134), (7, 173440, 0.5634977669624

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. CONFIDENCE LABELLING
#    HIGH   (≥0.65) → direct value borrowing (food-specific USDA values)
#    MEDIUM (≥0.40) → borrow with flagging; recommend manual review
#    LOW    (<0.40) → category-level borrowing only (use USDA category mean)
# ─────────────────────────────────────────────────────────────────────────────
def classify_confidence(score: float) -> str:
    if score >= 0.65: return "HIGH"
    if score >= 0.40: return "MEDIUM"
    return "LOW"
 
for df in [tkpi_links, myfcd_links]:
    df["confidence"] = df["similarity_score"].apply(classify_confidence)
 
# Keep only the best match per food
tkpi_best  = (tkpi_links
              .sort_values("similarity_score", ascending=False)
              .drop_duplicates("source_food_id"))
myfcd_best = (myfcd_links
              .sort_values("similarity_score", ascending=False)
              .drop_duplicates("source_food_id"))
 
print(f"\nTKPI  matched: {len(tkpi_best)} / {len(tkpi)} foods")
print(f"MyFCD matched: {len(myfcd_best)} / {len(myfcd)} foods")
print("\nTKPI confidence:\n",  tkpi_best["confidence"].value_counts())
print("\nMyFCD confidence:\n", myfcd_best["confidence"].value_counts())


TKPI  matched: 334 / 1146 foods
MyFCD matched: 138 / 234 foods

TKPI confidence:
 confidence
LOW       198
MEDIUM    118
HIGH       18
Name: count, dtype: int64

MyFCD confidence:
 confidence
LOW       89
MEDIUM    32
HIGH      17
Name: count, dtype: int64


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# 7. VALUE BORROWING
#    Nutrients present in USDA but absent from TKPI/MyFCD schema:
#      magnesium_mg, sugars_total_g, saturated_fat_g,
#      cholesterol_mg, vitamin_b6_mg, vitamin_b12_mcg
#
#    BORROWING POLICY (based on food type + confidence):
#
#    ┌──────────────┬────────────────────────────┬────────────────────────┐
#    │ Confidence   │ Food type                  │ Action                 │
#    ├──────────────┼────────────────────────────┼────────────────────────┤
#    │ HIGH (≥0.65) │ Raw/unprocessed            │ Direct food-level borrow│
#    │ HIGH (≥0.65) │ Cooked/processed           │ Borrow + flag warning  │
#    │ MEDIUM       │ Any                        │ Borrow + flag + review │
#    │ LOW (<0.40)  │ Any                        │ Category mean only     │
#    └──────────────┴────────────────────────────┴────────────────────────┘
# ─────────────────────────────────────────────────────────────────────────────
TARGET_NUTRIENTS = {
    "Magnesium, Mg":                  "magnesium_mg",
    "Sugars, Total":                  "sugars_total_g",
    "Fatty acids, total saturated":   "saturated_fat_g",
    "Cholesterol":                    "cholesterol_mg",
    "Vitamin B-6":                    "vitamin_b6_mg",
    "Vitamin B-12":                   "vitamin_b12_mcg",
}
 
target_nutrient_names = list(TARGET_NUTRIENTS.keys())
 
# Build a lookup: fdc_id → {nutrient_name: amount}
nutrient_map = ( 
    food_nutrient.merge(nutrient_meta[["id", "name"]], left_on="nutrient_id", right_on="id") 
    .query("name in @target_nutrient_names") 
    [["fdc_id", "name", "amount"]] 
)
usda_pivot = nutrient_map.pivot_table(
    index="fdc_id", columns="name", values="amount", aggfunc="first"
).rename(columns=TARGET_NUTRIENTS)
 
# Category-level means (fallback for LOW-confidence matches)
# Step 1: attach USDA category to usda_pivot
usda_pivot_cat = usda_pivot.merge(
    usda[["fdc_id", "usda_category"]], on="fdc_id", how="left"
)
cat_means = usda_pivot_cat.groupby("usda_category")[list(TARGET_NUTRIENTS.values())].mean()
 
 
def borrow_values(best_links: pd.DataFrame, source_df: pd.DataFrame,
                  source_id_col: str) -> pd.DataFrame:
    """
    Add new nutrient columns to source_df by borrowing from USDA.
    Returns source_df with new columns appended.
    """
    out = source_df.copy()
    new_cols = list(TARGET_NUTRIENTS.values())
    for col in new_cols:
        out[col]                  = np.nan
        out[col + "_source"]      = "not_matched"   # provenance flag
        out[col + "_confidence"]  = np.nan
 
    for _, link_row in best_links.iterrows():
        src_id     = link_row["source_food_id"]
        fdc_id     = link_row["usda_fdc_id"]
        confidence = link_row["confidence"]
        usda_cat   = link_row["usda_category"]
        mask       = out[source_id_col] == src_id
 
        if confidence in ("HIGH", "MEDIUM"):
            # Food-level borrow: pull exact USDA values for this food
            if fdc_id in usda_pivot.index:
                for col in new_cols:
                    val = usda_pivot.loc[fdc_id, col] if col in usda_pivot.columns else np.nan
                    out.loc[mask, col] = val
                    out.loc[mask, col + "_source"]     = f"USDA_direct_{confidence.lower()}"
                    out.loc[mask, col + "_confidence"] = link_row["similarity_score"]
 
        else:  # LOW confidence → category mean
            if usda_cat in cat_means.index:
                for col in new_cols:
                    val = cat_means.loc[usda_cat, col] if col in cat_means.columns else np.nan
                    out.loc[mask, col] = val
                    out.loc[mask, col + "_source"]     = "USDA_category_mean"
                    out.loc[mask, col + "_confidence"] = link_row["similarity_score"]
 
    return out
 
print("\nBorrowing USDA values into TKPI ...")
tkpi_enriched  = borrow_values(tkpi_best,  tkpi,  "food_id")
print("Borrowing USDA values into MyFCD ...")
myfcd_enriched = borrow_values(myfcd_best, myfcd, "food_id")


Borrowing USDA values into TKPI ...
Borrowing USDA values into MyFCD ...


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# 8. SAVE OUTPUTS
# ─────────────────────────────────────────────────────────────────────────────
tkpi_best.to_csv( f"{OUTPUT_DIR}/tkpi_usda_links.csv",   index=False)
myfcd_best.to_csv(f"{OUTPUT_DIR}/myfcd_usda_links.csv",  index=False)
tkpi_enriched.to_csv( f"{OUTPUT_DIR}/tkpi_enriched.csv",  index=False)
myfcd_enriched.to_csv(f"{OUTPUT_DIR}/myfcd_enriched.csv", index=False)
print("\nOutputs saved:")
print("  tkpi_usda_links.csv   — linkage table")
print("  myfcd_usda_links.csv  — linkage table")
print("  tkpi_enriched.csv     — TKPI + 6 new USDA nutrient columns")
print("  myfcd_enriched.csv    — MyFCD + 6 new USDA nutrient columns")


Outputs saved:
  tkpi_usda_links.csv   — linkage table
  myfcd_usda_links.csv  — linkage table
  tkpi_enriched.csv     — TKPI + 6 new USDA nutrient columns
  myfcd_enriched.csv    — MyFCD + 6 new USDA nutrient columns


In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# 9. QUICK VALIDATION — Kacang Arab (chickpea) example
# ─────────────────────────────────────────────────────────────────────────────
print("\n=== Validation: Kacang Arab (tkpi_0245) ===")
row = tkpi_enriched[tkpi_enriched["food_id"] == "tkpi_0245"]
new_cols = list(TARGET_NUTRIENTS.values())
src_cols = [c + "_source" for c in new_cols]
print(row[["food_id","food_name_normalized"] + new_cols + src_cols].T.to_string())


=== Validation: Kacang Arab (tkpi_0245) ===
                                                         244
food_id                                            tkpi_0245
food_name_normalized    kacang arab, kering (chick pea, raw)
magnesium_mg                                             NaN
sugars_total_g                                           NaN
saturated_fat_g                                          NaN
cholesterol_mg                                           NaN
vitamin_b6_mg                                            NaN
vitamin_b12_mcg                                          NaN
magnesium_mg_source                              not_matched
sugars_total_g_source                            not_matched
saturated_fat_g_source                           not_matched
cholesterol_mg_source                            not_matched
vitamin_b6_mg_source                             not_matched
vitamin_b12_mcg_source                           not_matched


In [15]:
# =============================================================================
# ---
# # THREE-WAY LINKAGE: TKPI ↔ MyFCD ↔ USDA
#
# Bagian ini adalah **tambahan** pada notebook asli. Tujuannya menghasilkan tabel
# linkage yang menghubungkan **TKPI dan MyFCD secara langsung**, dengan output utama
# berupa tiga kolom: `tkpi_food_id`, `myfcd_food_id`, `similarity_score`.
#
# Ada dua pendekatan yang disediakan:
#
# 1. **Direct matching (utama)** — mencocokkan nama pangan TKPI ↔ MyFCD secara langsung
#    menggunakan TF-IDF cosine dalam blok kelompok pangan yang sama. Lebih akurat karena
#    membandingkan nama SEA dengan nama SEA.
# 2. **USDA-bridge (cross-check)** — dua pangan yang tertaut ke `fdc_id` USDA yang sama
#    dianggap saling terkait secara transitif. Berguna sebagai validasi silang.
#
# Selain itu dihasilkan pula **non-match list**: pangan yang unik pada masing-masing
# basis data (tidak memiliki padanan di atas ambang).
# =============================================================================
 
# =============================================================================
# ## 10. Crosswalk Kelompok Pangan Bersama (TKPI ↔ MyFCD)
#
# TKPI dan MyFCD memakai taksonomi kategori yang berbeda. Agar dapat di-*blocking*
# secara langsung, kedua taksonomi dipetakan ke satu set **kelompok pangan bersama**
# (`_group`). Pemetaan ini konsisten dengan crosswalk ke USDA yang sudah ada.
# =============================================================================
 
# ─────────────────────────────────────────────────────────────────────────────
# 10. SHARED FOOD-GROUP CROSSWALK  (TKPI ↔ MyFCD direct blocking)
#     Map both regional taxonomies to a common coarse food group so that
#     TKPI and MyFCD items can be compared directly within the same block.
# ─────────────────────────────────────────────────────────────────────────────
TKPI_TO_GROUP = {
    "serealia":              "cereals",
    "umbi berpati":          "cereals",
    "sayuran":               "vegetables",
    "buah":                  "fruits",
    "daging":                "meat",
    "ikan/kerang/udang dll": "fish",
    "kacang-kacangan":       "legumes_nuts",
    "telur":                 "dairy_egg",
    "susu":                  "dairy_egg",
    "minyak/lemak":          "fats",
    "bumbu":                 "misc",
    "minuman non alkohol":   "beverages",
    "konfeksioneri":         "sweets",
}
 
MYFCD_TO_GROUP = {
    "cereals and grain products":         "cereals",
    "rice and rice flour based":          "cereals",
    "wheat flour based":                  "cereals",
    "cereal based":                       "cereals",
    "glutinous rice based":               "cereals",
    "porridge and ipengat/i":             "cereals",
    "starchy roots, tubers and products": "cereals",
    "tuber based":                        "cereals",
    "vegetables and vegetable products":  "vegetables",
    "vegetable dishes":                   "vegetables",
    "vegetables and fruits based":        "vegetables",
    "fruits and fruit products":          "fruits",
    "meat and meat products":             "meat",
    "meat dishes":                        "meat",
    "fish, shellfish and products":       "fish",
    "fish and sea-food dishes":           "fish",
    "legumes and legume products":        "legumes_nuts",
    "legume based":                       "legumes_nuts",
    "nuts, seeds and products":           "legumes_nuts",
    "eggs":                               "dairy_egg",
    "milk and milk products":             "dairy_egg",
    "oils and fats":                      "fats",
    "sugars and syrups":                  "sweets",
    "beverages":                          "beverages",
    "miscellaneous":                      "misc",
}
 
tkpi["_group"]  = tkpi["category_normalized"].map(TKPI_TO_GROUP)
myfcd["_group"] = myfcd["category_normalized"].map(MYFCD_TO_GROUP)
 
# Report any categories that failed to map (so you can extend the dicts)
unmapped_tkpi  = tkpi.loc[tkpi["_group"].isna(),  "category_normalized"].unique()
unmapped_myfcd = myfcd.loc[myfcd["_group"].isna(), "category_normalized"].unique()
if len(unmapped_tkpi):
    print("[warn] TKPI categories not mapped to a group:", list(unmapped_tkpi))
if len(unmapped_myfcd):
    print("[warn] MyFCD categories not mapped to a group:", list(unmapped_myfcd))
print("Shared groups present in both:",
      sorted(set(tkpi["_group"].dropna()) & set(myfcd["_group"].dropna())))

Shared groups present in both: ['beverages', 'cereals', 'dairy_egg', 'fats', 'fish', 'fruits', 'legumes_nuts', 'meat', 'misc', 'sweets', 'vegetables']


In [16]:
# =============================================================================
# ## 11. Direct TKPI ↔ MyFCD Matching (pendekatan utama)
#
# Menggunakan kembali `tfidf_cosine_match()` dari notebook asli, tetapi kini
# korpusnya adalah MyFCD (bukan USDA). Untuk setiap pangan TKPI dicari padanan
# MyFCD terbaik dalam blok kelompok yang sama.
# =============================================================================
 
# ─────────────────────────────────────────────────────────────────────────────
# 11. DIRECT TKPI ↔ MyFCD MATCHING
# ─────────────────────────────────────────────────────────────────────────────
def match_tkpi_to_myfcd(tkpi_df, myfcd_df, threshold=0.20, top_n=3):
    """
    For each TKPI food, find its top-n MyFCD matches within the same shared
    food group. Returns all pairs above threshold (long format).
    """
    all_links = []
    shared_groups = sorted(set(tkpi_df["_group"].dropna()) &
                           set(myfcd_df["_group"].dropna()))
    for grp in shared_groups:
        t_block = tkpi_df[tkpi_df["_group"] == grp]
        m_block = myfcd_df[myfcd_df["_group"] == grp]
        if t_block.empty or m_block.empty:
            continue
 
        q_names = t_block["name_norm"].tolist()
        c_names = m_block["name_norm"].tolist()
        c_ids   = m_block["food_id"].tolist()
        t_ids   = t_block["food_id"].tolist()
        t_orig  = t_block["food_name_normalized"].tolist()
 
        matches = tfidf_cosine_match(q_names, c_names, c_ids, top_n=top_n)
        for qi, myfcd_id, score in matches:
            if score < threshold:
                continue
            m_row = myfcd_df[myfcd_df["food_id"] == myfcd_id].iloc[0]
            all_links.append({
                "tkpi_food_id":     t_ids[qi],
                "tkpi_food_name":   t_orig[qi],
                "myfcd_food_id":    myfcd_id,
                "myfcd_food_name":  m_row["food_name_normalized"],
                "food_group":       grp,
                "similarity_score": round(score, 4),
            })
    return pd.DataFrame(all_links)
 
print("Matching TKPI ↔ MyFCD (direct) ...")
tm_links = match_tkpi_to_myfcd(tkpi, myfcd, threshold=0.20, top_n=3)
 
# Add confidence label (reuse classify_confidence from Cell 6)
tm_links["confidence"] = tm_links["similarity_score"].apply(classify_confidence)
 
# Keep the single best MyFCD match per TKPI food
tm_best = (tm_links
           .sort_values("similarity_score", ascending=False)
           .drop_duplicates("tkpi_food_id")
           .reset_index(drop=True))
 
print(f"\nTKPI foods with a MyFCD match: {tm_best['tkpi_food_id'].nunique()} / {len(tkpi)}")
print(f"Distinct MyFCD foods used:      {tm_best['myfcd_food_id'].nunique()} / {len(myfcd)}")
print("\nConfidence distribution:\n", tm_best["confidence"].value_counts())
tm_best.head(10)

Matching TKPI ↔ MyFCD (direct) ...

TKPI foods with a MyFCD match: 156 / 1146
Distinct MyFCD foods used:      55 / 234

Confidence distribution:
 confidence
LOW       124
MEDIUM     27
HIGH        5
Name: count, dtype: int64


,tkpi_food_id,tkpi_food_name,myfcd_food_id,myfcd_food_name,food_group,similarity_score,confidence
0,tkpi_1083,minyak kelapa (coconut oil),myfcd_0110,"oil, coconut (minyak kelapa)",fats,1.0000,HIGH
1,tkpi_0390,"bayam, segar (spinach, fresh)",myfcd_0228,"spinach, (bayam masak air)",vegetables,0.8102,HIGH
2,tkpi_0467,"jagung muda / semi, segar (baby corn, fresh)",myfcd_0035,baby corn (jagung sayur),vegetables,0.7385,HIGH
3,tkpi_0833,"rendang sapi, masakan",myfcd_0215,beef rendang (rendang daging lembu),meat,0.7022,HIGH
4,tkpi_1100,madu (honey),myfcd_0060,honey,sweets,0.6696,HIGH
5,tkpi_1080,"minyak ikan (fish, oil)",myfcd_0104,"fish oil, cod liver oil (minyak ikan kod)",fats,0.6422,MEDIUM
6,tkpi_0933,"kerang, segar",myfcd_0221,"cockles, boiled (kerang rebus)",fish,0.6407,MEDIUM
7,tkpi_0621,"buah naga merah, segar",myfcd_0040,"dragon fruit, red (buah naga, isi merah)",fruits,0.6122,MEDIUM
8,tkpi_0622,"buah naga putih, segar",myfcd_0041,"dragon fruit, white (buah naga, isi putih)",fruits,0.6112,MEDIUM
9,tkpi_0391,"bayam merah, segar (spinach, red, fresh)",myfcd_0228,"spinach, (bayam masak air)",vegetables,0.5893,MEDIUM


In [17]:
# =============================================================================
# ## 12. Tabel Linkage 3 Kolom (output yang diminta)
#
# Output inti: `tkpi_food_id`, `myfcd_food_id`, `similarity_score`.
# =============================================================================
 
# ─────────────────────────────────────────────────────────────────────────────
# 12. THREE-COLUMN LINKAGE TABLE
# ─────────────────────────────────────────────────────────────────────────────
tkpi_myfcd_links = tm_best[["tkpi_food_id", "myfcd_food_id", "similarity_score"]].copy()
 
print("tkpi_myfcd_links (3 kolom):", tkpi_myfcd_links.shape)
tkpi_myfcd_links.head(10)

tkpi_myfcd_links (3 kolom): (156, 3)


,tkpi_food_id,myfcd_food_id,similarity_score
0,tkpi_1083,myfcd_0110,1.0000
1,tkpi_0390,myfcd_0228,0.8102
2,tkpi_0467,myfcd_0035,0.7385
3,tkpi_0833,myfcd_0215,0.7022
4,tkpi_1100,myfcd_0060,0.6696
5,tkpi_1080,myfcd_0104,0.6422
6,tkpi_0933,myfcd_0221,0.6407
7,tkpi_0621,myfcd_0040,0.6122
8,tkpi_0622,myfcd_0041,0.6112
9,tkpi_0391,myfcd_0228,0.5893


In [18]:
# =============================================================================
# ## 12b. (Opsional) Reciprocal / Mutual-Best Matching
#
# Pada tabel di atas, beberapa pangan TKPI dapat menunjuk ke pangan MyFCD yang sama
# (many-to-one). Untuk linkage berpresisi lebih tinggi, gunakan **mutual best match**:
# sepasang (TKPI, MyFCD) dipertahankan hanya jika MyFCD adalah padanan terbaik bagi
# TKPI **dan** sebaliknya TKPI adalah padanan terbaik bagi MyFCD. Ini menghasilkan
# tautan 1-1 yang lebih dapat dipercaya (cocok untuk pelaporan konservatif).
# =============================================================================
 
# ─────────────────────────────────────────────────────────────────────────────
# 12b. RECIPROCAL (MUTUAL-BEST) MATCHING  — optional, higher precision
# ─────────────────────────────────────────────────────────────────────────────
# Best MyFCD match for each TKPI food (already computed: tm_best)
# Now compute best TKPI match for each MyFCD food (reverse direction).
def match_myfcd_to_tkpi(tkpi_df, myfcd_df, threshold=0.20, top_n=3):
    all_links = []
    shared_groups = sorted(set(tkpi_df["_group"].dropna()) &
                           set(myfcd_df["_group"].dropna()))
    for grp in shared_groups:
        t_block = tkpi_df[tkpi_df["_group"] == grp]
        m_block = myfcd_df[myfcd_df["_group"] == grp]
        if t_block.empty or m_block.empty:
            continue
        # queries = MyFCD, corpus = TKPI
        matches = tfidf_cosine_match(m_block["name_norm"].tolist(),
                                     t_block["name_norm"].tolist(),
                                     t_block["food_id"].tolist(), top_n=top_n)
        m_ids = m_block["food_id"].tolist()
        for qi, tkpi_id, score in matches:
            if score < threshold:
                continue
            all_links.append({"myfcd_food_id": m_ids[qi],
                              "tkpi_food_id": tkpi_id,
                              "similarity_score": round(score, 4)})
    return pd.DataFrame(all_links)
 
mt_links = match_myfcd_to_tkpi(tkpi, myfcd, threshold=0.20, top_n=3)
mt_best = (mt_links.sort_values("similarity_score", ascending=False)
           .drop_duplicates("myfcd_food_id"))
 
# Mutual-best: pairs that agree in both directions
fwd_pairs = set(zip(tm_best["tkpi_food_id"], tm_best["myfcd_food_id"]))
rev_pairs = set(zip(mt_best["tkpi_food_id"], mt_best["myfcd_food_id"]))
mutual = fwd_pairs & rev_pairs
 
tkpi_myfcd_links_mutual = (
    tm_best[tm_best.apply(lambda r: (r["tkpi_food_id"], r["myfcd_food_id"]) in mutual, axis=1)]
    [["tkpi_food_id", "myfcd_food_id", "similarity_score", "confidence"]]
    .reset_index(drop=True)
)
 
print(f"Mutual-best (1-1) links: {len(tkpi_myfcd_links_mutual)}")
print("Confidence distribution:\n", tkpi_myfcd_links_mutual["confidence"].value_counts())
tkpi_myfcd_links_mutual.head(10)

Mutual-best (1-1) links: 52
Confidence distribution:
 confidence
LOW       30
MEDIUM    17
HIGH       5
Name: count, dtype: int64


,tkpi_food_id,myfcd_food_id,similarity_score,confidence
0,tkpi_1083,myfcd_0110,1.0000,HIGH
1,tkpi_0390,myfcd_0228,0.8102,HIGH
2,tkpi_0467,myfcd_0035,0.7385,HIGH
3,tkpi_0833,myfcd_0215,0.7022,HIGH
4,tkpi_1100,myfcd_0060,0.6696,HIGH
5,tkpi_1080,myfcd_0104,0.6422,MEDIUM
6,tkpi_0933,myfcd_0221,0.6407,MEDIUM
7,tkpi_0621,myfcd_0040,0.6122,MEDIUM
8,tkpi_0622,myfcd_0041,0.6112,MEDIUM
9,tkpi_0259,myfcd_0030,0.5878,MEDIUM


In [19]:
# =============================================================================
# ## 13. USDA-Bridge Cross-Check (opsional)
#
# Dua pangan yang tertaut ke `fdc_id` USDA yang **sama** dianggap saling terkait
# secara transitif (TKPI → USDA ← MyFCD). Skor jembatan diambil sebagai nilai
# terkecil (paling konservatif) dari kedua skor tautan ke USDA.
#
# Ini berguna untuk memvalidasi pasangan hasil direct matching, atau menemukan
# pasangan tambahan yang mungkin terlewat.
# =============================================================================
 
# ─────────────────────────────────────────────────────────────────────────────
# 13. USDA-BRIDGE CROSS-CHECK  (transitive linkage via shared fdc_id)
# ─────────────────────────────────────────────────────────────────────────────
# tkpi_best / myfcd_best come from Cell 6 (best USDA match per source food).
bridge = tkpi_best[["source_food_id", "usda_fdc_id", "similarity_score"]].rename(
    columns={"source_food_id": "tkpi_food_id",
             "similarity_score": "tkpi_usda_score"}
).merge(
    myfcd_best[["source_food_id", "usda_fdc_id", "similarity_score"]].rename(
        columns={"source_food_id": "myfcd_food_id",
                 "similarity_score": "myfcd_usda_score"}),
    on="usda_fdc_id", how="inner"
)
 
# Bridge score = min of the two link scores (conservative)
bridge["bridge_score"] = bridge[["tkpi_usda_score", "myfcd_usda_score"]].min(axis=1)
bridge = (bridge
          .sort_values("bridge_score", ascending=False)
          .drop_duplicates(["tkpi_food_id", "myfcd_food_id"])
          .reset_index(drop=True))
 
print(f"USDA-bridge candidate pairs: {len(bridge)}")
print(f"Distinct TKPI foods bridged: {bridge['tkpi_food_id'].nunique()}")
 
# Cross-check: which direct matches are corroborated by the USDA bridge?
direct_pairs = set(zip(tkpi_myfcd_links["tkpi_food_id"],
                       tkpi_myfcd_links["myfcd_food_id"]))
bridge_pairs = set(zip(bridge["tkpi_food_id"], bridge["myfcd_food_id"]))
corroborated = direct_pairs & bridge_pairs
print(f"Direct matches also supported by USDA bridge: {len(corroborated)}")
 
bridge.head(10)

USDA-bridge candidate pairs: 51
Distinct TKPI foods bridged: 36
Direct matches also supported by USDA bridge: 24


,tkpi_food_id,usda_fdc_id,tkpi_usda_score,myfcd_food_id,myfcd_usda_score,bridge_score
0,tkpi_1100,169640,0.6298,myfcd_0060,1.0000,0.6298
1,tkpi_1083,171412,0.5625,myfcd_0110,0.5489,0.5489
2,tkpi_0426,169975,0.5036,myfcd_0225,0.5503,0.5036
3,tkpi_0391,168462,0.4611,myfcd_0228,0.4390,0.4390
4,tkpi_0390,168462,0.6364,myfcd_0228,0.4390,0.4390
5,tkpi_1079,173577,0.4136,myfcd_0104,0.6958,0.4136
6,tkpi_0163,170542,0.4047,myfcd_0024,0.3825,0.3825
7,tkpi_0463,170542,0.3847,myfcd_0024,0.3825,0.3825
8,tkpi_0460,170542,0.3940,myfcd_0024,0.3825,0.3825
9,tkpi_0166,170542,0.3949,myfcd_0024,0.3825,0.3825


In [20]:
# =============================================================================
# ## 14. Non-Match Lists (pangan unik per basis data)
#
# Pangan yang **tidak** memiliki padanan di atas ambang pada tabel direct linkage.
# Ini penting untuk narasi riset: item unik menunjukkan cakupan khas masing-masing
# basis data (mis. pangan lokal Indonesia yang tak ada padanannya di Malaysia).
# =============================================================================
 
# ─────────────────────────────────────────────────────────────────────────────
# 14. NON-MATCH LISTS
# ─────────────────────────────────────────────────────────────────────────────
matched_tkpi_ids  = set(tkpi_myfcd_links["tkpi_food_id"])
matched_myfcd_ids = set(tkpi_myfcd_links["myfcd_food_id"])
 
tkpi_only = tkpi.loc[~tkpi["food_id"].isin(matched_tkpi_ids),
                     ["food_id", "food_name_original",
                      "category_normalized", "_group"]].copy()
myfcd_only = myfcd.loc[~myfcd["food_id"].isin(matched_myfcd_ids),
                       ["food_id", "food_name_original",
                        "category_normalized", "_group"]].copy()
 
tkpi_only  = tkpi_only.rename(columns={"food_id": "tkpi_food_id"})
myfcd_only = myfcd_only.rename(columns={"food_id": "myfcd_food_id"})
 
print(f"TKPI-only  (unique foods): {len(tkpi_only)} / {len(tkpi)}")
print(f"MyFCD-only (unique foods): {len(myfcd_only)} / {len(myfcd)}")
 
print("\nTKPI-only by group:\n",  tkpi_only["_group"].value_counts())
print("\nMyFCD-only by group:\n", myfcd_only["_group"].value_counts())
 
tkpi_only.head(10)

TKPI-only  (unique foods): 990 / 1146
MyFCD-only (unique foods): 179 / 234

TKPI-only by group:
 _group
vegetables      202
cereals         177
fish            163
legumes_nuts    128
fruits          124
meat            115
misc             32
dairy_egg        29
sweets           12
fats              7
beverages         1
Name: count, dtype: int64

MyFCD-only by group:
 _group
cereals         63
dairy_egg       27
misc            26
sweets          21
beverages       19
fats             8
vegetables       4
legumes_nuts     3
fruits           3
meat             3
fish             2
Name: count, dtype: int64


,tkpi_food_id,food_name_original,category_normalized,_group
7,tkpi_0008,"Beras ketan putih tumbuk, mentah (Glutinous ri...",serealia,cereals
10,tkpi_0011,"Beras parboiled (Rice, parboiled)",serealia,cereals
13,tkpi_0014,"Cantel, mentah (Sorghum, raw)",serealia,cereals
18,tkpi_0019,"Jali, mentah (Job's tear/adlay, raw)",serealia,cereals
19,tkpi_0020,"Jawawut, mentah (Italian millet, raw)",serealia,cereals
20,tkpi_0021,"Jampang huma, mentah (Ragi millet, raw)",serealia,cereals
22,tkpi_0023,Nasi tim,serealia,cereals
26,tkpi_0027,"Bihun, mentah",serealia,cereals
27,tkpi_0028,Bihun goreng instan,serealia,cereals
28,tkpi_0029,"Bihun Jagung, mentah",serealia,cereals


In [21]:
# =============================================================================
# ## 15. Simpan Output Three-Way
#
# Menyimpan tabel linkage 3 kolom, versi lengkapnya (dengan nama & confidence),
# jembatan USDA, dan kedua non-match list.
# =============================================================================
 
# ─────────────────────────────────────────────────────────────────────────────
# 15. SAVE THREE-WAY OUTPUTS
# ─────────────────────────────────────────────────────────────────────────────
# Core requested output: 3-column linkage table
tkpi_myfcd_links.to_csv(f"{OUTPUT_DIR}/tkpi_myfcd_links.csv", index=False)
tkpi_myfcd_links_mutual.to_csv(f"{OUTPUT_DIR}/tkpi_myfcd_links_mutual.csv", index=False)
 
# Detailed version (names + group + confidence) for auditing
tm_best.to_csv(f"{OUTPUT_DIR}/tkpi_myfcd_links_detailed.csv", index=False)
 
# USDA-bridge cross-check
bridge.to_csv(f"{OUTPUT_DIR}/tkpi_myfcd_bridge_via_usda.csv", index=False)
 
# Non-match lists
tkpi_only.to_csv(f"{OUTPUT_DIR}/tkpi_only_nonmatch.csv",  index=False)
myfcd_only.to_csv(f"{OUTPUT_DIR}/myfcd_only_nonmatch.csv", index=False)
 
print("Three-way outputs saved:")
print("  tkpi_myfcd_links.csv           — 3 kolom: tkpi_food_id, myfcd_food_id, similarity_score")
print("  tkpi_myfcd_links_detailed.csv  — versi lengkap (nama, grup, confidence)")
print("  tkpi_myfcd_bridge_via_usda.csv — cross-check jembatan USDA")
print("  tkpi_only_nonmatch.csv         — pangan unik TKPI")
print("  myfcd_only_nonmatch.csv        — pangan unik MyFCD")

Three-way outputs saved:
  tkpi_myfcd_links.csv           — 3 kolom: tkpi_food_id, myfcd_food_id, similarity_score
  tkpi_myfcd_links_detailed.csv  — versi lengkap (nama, grup, confidence)
  tkpi_myfcd_bridge_via_usda.csv — cross-check jembatan USDA
  tkpi_only_nonmatch.csv         — pangan unik TKPI
  myfcd_only_nonmatch.csv        — pangan unik MyFCD


In [22]:
# =============================================================================
# ## 16. Ringkasan Integrasi Tiga Basis Data
#
# Tabel ringkas status penautan untuk narasi Bab 3.
# =============================================================================
 
# ─────────────────────────────────────────────────────────────────────────────
# 16. INTEGRATION SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
summary = pd.DataFrame([
    {"pair": "TKPI → USDA",   "matched": tkpi_best["source_food_id"].nunique(),
     "total": len(tkpi),  "pct": round(100*tkpi_best["source_food_id"].nunique()/len(tkpi), 1)},
    {"pair": "MyFCD → USDA",  "matched": myfcd_best["source_food_id"].nunique(),
     "total": len(myfcd), "pct": round(100*myfcd_best["source_food_id"].nunique()/len(myfcd), 1)},
    {"pair": "TKPI ↔ MyFCD (direct)", "matched": tkpi_myfcd_links["tkpi_food_id"].nunique(),
     "total": len(tkpi),  "pct": round(100*tkpi_myfcd_links["tkpi_food_id"].nunique()/len(tkpi), 1)},
])
print("=== Integration summary ===")
print(summary.to_string(index=False))
 
print(f"\nTKPI-only (no MyFCD match):  {len(tkpi_only)}")
print(f"MyFCD-only (no TKPI match):  {len(myfcd_only)}")
print(f"USDA-bridge corroborated:    {len(corroborated)} direct pairs")

=== Integration summary ===
                 pair  matched  total  pct
          TKPI → USDA      334   1146 29.1
         MyFCD → USDA      138    234 59.0
TKPI ↔ MyFCD (direct)      156   1146 13.6

TKPI-only (no MyFCD match):  990
MyFCD-only (no TKPI match):  179
USDA-bridge corroborated:    24 direct pairs
